In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from flipper_training.experiments.ppo.policy_inference_module import PPOPolicyInferenceModule

/Users/davidkorcak/.venv/lib/python3.12/site-packages/torchrl/__init__.py:43: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(


In [3]:
config_path = "../test_configs/average_ensembled_policy.yaml"
weights_path = "../modified_networks/top_3_averaged/policy.pth"
vecnorm_path = "../modified_networks/top_3_averaged/vecnorm.pth"

In [4]:
inference_module = PPOPolicyInferenceModule(
    train_config_path=config_path,
    policy_weights_path=weights_path,
    vecnorm_weights_path=vecnorm_path,
    device="cpu",
)

2026-01-09 11:10:21,249 [RobotModelConfig][INFO]: Loading robot model from cache: /Users/davidkorcak/Documents/ctu/bachelors/flipper_training/.robot_cache/marv_vx0.010_dp384_b512_whl0.02_trck0.05_eaecc2d5466de1eb8911703837d75c759b5c075158ced88ea318e932700dabb2 (robot_config.py:155)


Initializing start/goal position cache: 100%|██████████| 1/1 [00:00<00:00, 505.83it/s]


                      Environment Summary                       
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                 Key ┃                                  Value ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    Number of robots │                                      1 │
│        Observations │            LocalStateVector, Heightmap │
│              Reward │ PotentialGoalWithPenaltiesConfigurable │
│           Objective │              RandomNavigationObjective │
│   Physics frequency │                              142.86 Hz │
│   Engine iters/step │                                      4 │
│ Effective frequency │                               35.71 Hz │
└─────────────────────┴────────────────────────────────────────┘

Initializing start/goal position cache: 100%|██████████| 1/1 [00:00<00:00, 378.21it/s]
2026-01-09 11:10:21,324 [torchrl][INFO] check_env_specs succeeded!


2026-01-09 11:10:21,334 [MLPPolicyConfig][INFO]: Applied orthogonal initialization to the actor and value operators. (mlp_policy.py:89)


...

     Policy Parameter Counts     
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃        Component ┃ Parameters ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│    Actor Encoder │     32,032 │
│       Actor Head │     13,456 │
│    Value Encoder │     32,032 │
│       Value Head │     12,737 │
│              --- │        --- │
│      Total Actor │     45,488 │
│      Total Value │     44,769 │
│              --- │        --- │
│ Total Parameters │     90,257 │
└──────────────────┴────────────┘

2026-01-09 11:10:21,345 [policy_inference_module][INFO]: Environment transforms: Compose(
        StepCounter(keys=[]),
        RawRewardSaveTransform(keys=['reward']),
        VecNorm(decay=0.9900,eps=0.0001, in_keys=['LocalStateVector', 'reward'], out_keys=['LocalStateVector', 'reward'])) (policy_inference_module.py:40)


INFO:policy_inference_module:Environment transforms: Compose(
        StepCounter(keys=[]),
        RawRewardSaveTransform(keys=['reward']),
        VecNorm(decay=0.9900,eps=0.0001, in_keys=['LocalStateVector', 'reward'], out_keys=['LocalStateVector', 'reward']))
/Users/davidkorcak/.venv/lib/python3.12/site-packages/torchrl/envs/transforms/transforms.py:7002: UserWarning: VecNorm wasn't initialized and the tensordict is not shared. In single process settings, this is ok, but if you need to share the statistics between workers this should require some attention. Make sure that the content of VecNorm is transmitted to the workers after calling load_state_dict and not before, as other workers may not have access to the loaded TensorDict.
  warnings.warn(


2026-01-09 11:10:21,465 [policy_inference_module][INFO]: Loaded VecNorm weights from ../modified_networks/top_3_averaged/vecnorm.pth (policy_inference_module.py:43)


INFO:policy_inference_module:Loaded VecNorm weights from ../modified_networks/top_3_averaged/vecnorm.pth


## create random dummy imputs to show what we need to pass to the module

In [5]:
import numpy as np

# 1. heightmap
heightmap = np.random.rand(128,128).astype(np.float32) # METRIC heightmap in robot's local frame -> top top left corner is +x, +y, top right corner is +x, -y, bottom left corner is -x, +y, bottom right corner is -x, -y
heightmap_extent = [2.0, 2.0, -2.0, -2.0]  # xmax, ymax, xmin, ymin of the heightmap in meters

# 2. robot state
goal_vec_local = np.array([1.0, 0.0, 0.0], dtype=np.float32)  # goal vector in robot's local frame -> x forward, y left, z up

xd_local = np.ones(3, dtype=np.float32) * 0.5  # linear velocity in robot's local frame -> x forward, y left, z up

omega_local = np.zeros(3, dtype=np.float32)  # angular velocity in robot in robot's local frame

thetas = np.full(4, np.pi / 4, dtype=np.float32)  # flipper angles -> front left, front right, rear left, rear right

quat = torch.tensor([ 0.35355339,  0.35355339, -0.14644661,  0.85355339], dtype=torch.float32)  # quaternion representing orientation in the ROS format (x,y,z,w)

action = inference_module.infer_action(
    heightmap=heightmap,
    heightmap_extent=heightmap_extent,
    goal_vec_local=goal_vec_local,
    xd_local=xd_local,
    omega_local=omega_local,
    thetas=thetas,
    quat=quat,
)

/Users/davidkorcak/Documents/ctu/bachelors/flipper_training/flipper_training/experiments/ppo/policy_inference_module.py:61: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  {k: torch.tensor(v, device=self.device).unsqueeze(0) for k, v in kwargs.items()},


In [6]:
%%timeit
action = inference_module.infer_action(
    heightmap=heightmap,
    heightmap_extent=heightmap_extent,
    goal_vec_local=goal_vec_local,
    xd_local=xd_local,
    omega_local=omega_local,
    thetas=thetas,
    quat=quat,
)

1.36 ms ± 53 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [7]:
action # corresponds to track velocities front left, front right, rear left, rear right and joint rotational velocities in radians per second in order front left, front right, rear left, rear right

array([ 0.3511158 ,  0.7616981 ,  0.39312577,  0.70001245, -0.19588083,
       -0.01867814,  0.4094977 , -0.21361299], dtype=float32)